#  Optimizing a Model with LLM Compressor

In this notebook, you'll:
1. Learn how **post-training quantization** works via full precision & compressed model comparisons
2. Use the `llm-compressor` library to apply a GPTQ recipe that produces a W4A16 quantized model
3. Test and evaluate the quantized model against the original

## What is LLM Compressor?

[llm-compressor](https://github.com/vllm-project/llm-compressor) is the production quantization toolkit from the vLLM project. It takes a trained model and reduces precision in a single pass, no retraining required.

The core API is **`oneshot`**: you give it a model, a calibration dataset (small sample of real inputs used to minimize quantization error), and a recipe describing how to quantize (e.g. GPTQ, W4A16). It produces a smaller model that can be served directly by [vLLM](https://github.com/vllm-project/vllm), an LLM inference engine that you'll use in the next lesson.

```python
oneshot(
    model="model-name",           # HuggingFace model ID
    dataset="dataset-name",       # Calibration dataset
    recipe=recipe,                # Quantization configuration
    output_dir="./output",        # Where to save
    num_calibration_samples=512,  # Samples for calibration
    max_seq_length=4096,          # Sequence length
)
```

The name "oneshot" reflects that this happens in a **single pass** over calibration data, no retraining required.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

!pip install -q -r requirements.txt

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MODEL_ID = "Qwen/Qwen3-0.6B"
QUANTIZED_ID = "RedHatAI/Qwen3-0.6B-quantized.w4a16"

MODEL_DIR = snapshot_download(MODEL_ID)
OUTPUT_DIR = snapshot_download(QUANTIZED_ID)

print(f"Base model:      {MODEL_ID} → {MODEL_DIR}")
print(f"Quantized model: {QUANTIZED_ID} → {OUTPUT_DIR}")

Base model:      Qwen3-0.6B
Quantized model: Qwen3-0.6B-W4A16


## Define the Recipe

A **recipe** tells LLM Compressor how to quantize. It's a list of modifiers: each one specifies an algorithm and settings we'll apply to the model.

### Available Modifiers

**Quantization Modifiers:**

| Modifier | Description |
|:--|:--|
| `GPTQModifier` | GPTQ algorithm: uses calibration data to find optimal quantization values |
| `AWQModifier` | Activation-aware Weight QuantizationL preserves salient weights |
| `AutoRoundModifier` | Intel's algorithm with learnable rounding/clipping |
| `QuantizationModifier` | Basic PTQ and QAT for simple use cases |

**Transform Modifiers** (for improving accuracy):

| Modifier | Description |
|:--|:--|
| `SmoothQuantModifier` | Smooths activations before quantization, often paired with GPTQ |
| `QuIPModifier` | Hadamard rotations to reduce outliers |
| `SpinQuantModifier` | SpinQuant-style rotations to even out weight distributions |

Modifiers can be chained: e.g. applying `SmoothQuantModifier` before `GPTQModifier` improves accuracy for W8A8 quantization.

### Quantization Schemes

The `scheme` parameter determines the bit-width for weights (W) and activations (A):

| Scheme | Weights | Activations | Quantized Layer Reduction | Quality Impact |
|:--|:--|:--|:--|:--|
| `W8A16` | 8-bit | 16-bit (FP16) | ~50% | Minimal |
| `W4A16` | 4-bit | 16-bit (FP16) | ~75% | Low–Moderate |
| `W8A8` | 8-bit | 8-bit | ~50% | Low |
| `W4A8` | 4-bit | 8-bit | ~75% | Moderate |

> **Note:** These reductions apply to the quantized layers only. The embedding and `lm_head` layers are kept at full precision, so total model size reduction depends on how large those layers are relative to the rest. For small models (~0.6B), expect ~40–50% total reduction with W4A16.

### Our Recipe: GPTQModifier with W4A16

| Parameter | Value | Why |
|:--|:--|:--|
| `scheme` | `W4A16` | 4-bit weights  |
| `targets` | `Linear` | Linear layers hold most parameters - biggest savings |
| `ignore` | `["lm_head"]` | Output layer maps to vocabulary - keep it precise |

In [ ]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

## Quantize the Model

### Why a calibration dataset?

GPTQ doesn't just round weights to lower precision, it uses a small set of real text to measure how each weight affects the model's output, then finds quantized values that minimize the error. This is what makes it more accurate than naive rounding.

The `dataset` parameter specifies what text to use for calibration. Here you'll use [WikiText-2](https://huggingface.co/datasets/mindchain/wikitext2), a standard benchmark of Wikipedia articles, the same dataset you'll use later for perplexity evaluation. You only need a few hundred samples as calibration is fast.

>**Note:** Since quantization can take several minutes and benefits from a GPU, we've already downloaded the pre-quantized model from HuggingFace (`RedHatAI/Qwen3-0.6B-quantized.w4a16`). The `if not os.path.isdir(OUTPUT_DIR)` check ensures you skip re-running quantization when the model is already cached, so you can move straight to evaluation. 


In [ ]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=4096,
        num_calibration_samples=256,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

## Compare Model Sizes

Let's see how much space quantization saves.

In [4]:
def folder_size(path):
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(nbytes):
    if nbytes < 1024**2:
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3:
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig = folder_size(MODEL_DIR)
size_q = folder_size(OUTPUT_DIR)
reduction = (1 - size_q / size_orig) * 100 if size_orig > 0 else 0

print("Model Size Comparison")
print("=" * 45)
print(f"Original (BF16):    {format_size(size_orig)}")
print(f"Quantized (W4A16):  {format_size(size_q)}")
print(f"Reduction:          {reduction:.0f}%")

Model Size Comparison
Original (BF16):    1.41 GB
Quantized (W4A16):  835.3 MB
Reduction:          42%


> **Note**: You might expect a 75% reduction since you went from 16-bit to 4-bit weights (4x smaller), but the actual reduction is 42%. The reason: only the **linear layer weights** are quantized to Int4. The rest of the model (including the LM head and normalization layers) stays at higher precision.
> So the 4x compression applies to the bulk of the parameters (the linear layers, which dominate the model), but the unquantized pieces pull the overall reduction down to ~42%. This ratio improves with larger models, where linear weights make up an even bigger share of total size — a 70B model quantized the same way gets much closer to the theoretical 4x.

## Test Both Models

Smaller files are only useful if the model still produces reasonable output. Let's generate text from both and compare using the Hugging Face [Transformers](https://huggingface.co/docs/transformers/en/index) library, starting with the original model **then** the quantized model.

In [5]:
prompt = "Machine learning is a branch of"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

models = {
    f"Base ({MODEL_ID})": AutoModelForCausalLM.from_pretrained(MODEL_DIR, device_map="cpu", dtype=torch.bfloat16),
    f"Quantized ({QUANTIZED_ID})": AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, device_map="cpu", dtype=torch.bfloat16),
}

for label, model in models.items():
    inputs = tokenizer(prompt, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    print(f"{label}")
    print(f"  Prompt:   {prompt}")
    print(f"  Response: {tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)}\n")

`torch_dtype` is deprecated! Use `dtype` instead!


Base Model (Qwen3-0.6B)
Prompt: Machine learning is a branch of
Response:  computer science that has become a cornerstone of modern data science. It is particularly powerful in scenarios where large volumes of data are involved, such as in machine learning applications, predictive modeling, and data analysis. However, it is also one of the most complex and challenging areas to understand and apply in the real


Compressing model: 196it [00:00, 3249.78it/s]


Quantized Model (Qwen3-0.6B-W4A16)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has been gaining popularity in recent years. It's used to predict future trends and make decisions based on data. One of the key areas in machine learning is the use of neural networks, which are based on the idea of neurons in the human brain. These networks are used to make predictions


## Perplexity Comparison

Side-by-side text gives intuition, but **perplexity** is the standard metric: it measures how well the model predicts text. Lower is better. If quantization has degraded the model, its perplexity will be noticeably higher.

In [7]:
from datasets import load_dataset

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

ppls = {}
for label, model in models.items():
    ppl = calculate_perplexity(model, tokenizer, test_data)
    ppls[label] = ppl
    print(f"  {label}: {ppl:.2f}")

labels = list(ppls.keys())
base_ppl, quant_ppl = ppls[labels[0]], ppls[labels[1]]
print(f"\nDifference: {quant_ppl - base_ppl:+.2f} ({(quant_ppl/base_ppl - 1)*100:+.1f}%)")
print("A small increase in perplexity is expected — the quantized layers use 4-bit weights.")

Loaded 4358 test samples


Quantized perplexity: 35.48


Base perplexity: 32.79


Perplexity Comparison
Base (BF16):      32.79
Quantized (W4A16): 35.48
Difference:       +2.70 (+8.2%)

A small increase in perplexity is expected — the quantized layers use 4-bit weights.


## Summary

In this notebook, you:

- Learned how the LLM Compressor **`oneshot`** applies post-training quantization with a GPTQ recipe
- Compared model sizes: W4A16 reduces weights from 16-bit to 4-bit
- Tested both models on the same prompt to verify output quality
- Measured **perplexity** to quantify the accuracy tradeoff